# Chapter 2 — Why Is Kenya Losing Forest?
**How has the cause of forest loss shifted by county over time?**

Author: Davis Mironga  
Data: Global Forest Watch — Tree cover loss by dominant driver, 2001–2024

---
Drivers tracked by GFW:
- `Commodity-driven deforestation` — permanent agricultural expansion
- `Shifting agriculture` — smallholder farming cycles
- `Forestry` — logging, timber extraction
- `Wildfire` — fire-driven loss
- `Urbanization` — settlement expansion
- `Unknown` — unclassified loss

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

DB_PATH = Path('../kenya_deforestation.db')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load Data

In [ ]:
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT
        subnational_1 AS county,
        year,
        driver,
        tree_cover_loss_ha AS loss_ha
    FROM gfw_loss_by_driver
    WHERE year BETWEEN 2001 AND 2024
    ORDER BY county, year, driver
""", conn)

conn.close()
print(f"Loaded {len(df):,} rows | {df['county'].nunique()} counties | {df['driver'].nunique()} drivers")
df.head()

## 2. National Driver Mix Over Time

In [ ]:
national = df.groupby(['year', 'driver'])['loss_ha'].sum().reset_index()

fig = px.area(
    national,
    x='year', y='loss_ha', color='driver',
    title='Kenya: Forest Loss by Driver (2001–2024)',
    labels={'loss_ha': 'Tree Cover Loss (ha)', 'year': 'Year', 'driver': 'Driver'},
    template='plotly_white'
)
fig.show()

## 3. Dominant Driver Per County

In [ ]:
dominant = (
    df.groupby(['county', 'driver'])['loss_ha']
    .sum()
    .reset_index()
    .sort_values('loss_ha', ascending=False)
    .drop_duplicates('county')
    .rename(columns={'driver': 'dominant_driver', 'loss_ha': 'total_loss_ha'})
)

print(dominant['dominant_driver'].value_counts())
dominant.head(10)

## 4. Driver Shift Analysis — Did the Dominant Driver Change?

Compare the leading driver in **2001–2012** vs **2013–2024** per county.

In [ ]:
def top_driver(group):
    return group.loc[group['loss_ha'].idxmax(), 'driver']

early = df[df['year'] <= 2012].groupby(['county', 'driver'])['loss_ha'].sum().reset_index()
late  = df[df['year'] >= 2013].groupby(['county', 'driver'])['loss_ha'].sum().reset_index()

early_top = early.sort_values('loss_ha', ascending=False).drop_duplicates('county')[['county', 'driver']].rename(columns={'driver': 'driver_2001_2012'})
late_top  = late.sort_values('loss_ha', ascending=False).drop_duplicates('county')[['county', 'driver']].rename(columns={'driver': 'driver_2013_2024'})

shift = early_top.merge(late_top, on='county')
shift['driver_changed'] = shift['driver_2001_2012'] != shift['driver_2013_2024']

print(f"Counties where dominant driver changed: {shift['driver_changed'].sum()} / {len(shift)}")
shift[shift['driver_changed']]

## 5. Heatmap — Loss by County and Driver

In [ ]:
pivot = df.groupby(['county', 'driver'])['loss_ha'].sum().unstack(fill_value=0)

# Focus on top 15 counties by total loss
top15 = df.groupby('county')['loss_ha'].sum().nlargest(15).index
pivot_top = pivot.loc[top15]

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot_top, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Tree Cover Loss (ha) by County and Driver — Top 15 Counties', fontsize=13)
ax.set_xlabel('Driver')
ax.set_ylabel('County')
plt.tight_layout()
plt.savefig('../data/outputs/ch2_driver_heatmap.png', dpi=150)
plt.show()

## 6. Community Context

| Community | County | Primary Driver |
|-----------|--------|----------------|
| Ogiek | Nakuru / Kericho (Mau Forest) | Commodity agriculture / logging |
| Sengwer | Elgeyo-Marakwet (Cherangany) | Forestry / conservation displacement |
| Pastoralists | Baringo, Tana River | Shifting agriculture / wildfire |

Cross-reference `dominant_driver` results above with these counties.

---
**Next:** Chapter 3 — Power BI dashboard (Who bears the cost?)  
**Data needed:** `data/raw/gfw_loss_by_driver.csv` loaded into the database.